# Data Preprocessing: Craigslist Used Cars

Raw data: 426,880 postings × 26 columns. Goal: a clean table ready for analysis.

## 1. Load data

Limits chosen from the raw-data profile:
- **Price:** $500–$150,000 (32,895 prices are 0, about 9,000 more are $1–$499, the max is $3.7 billion).
- **Year:** 1950–2022 (1900 is a placeholder; 2022 is valid for 2021 listings).
- **Odometer:** up to 500,000 miles (99th percentile is 280k; the max is 10,000,000).

In [ ]:
import glob
import pandas as pd

PRICE_MIN, PRICE_MAX = 500, 150_000
YEAR_MIN, YEAR_MAX = 1950, 2022
ODO_MAX = 500_000

path = (glob.glob("/kaggle/input/**/vehicles.csv", recursive=True) or glob.glob("vehicles.csv"))[0]
df = pd.read_csv(path)
print("Raw shape:", df.shape)

## 2. Drop unused columns and clean text

- Dropped: `id` (a row number added when scraping, so it cannot identify a duplicate), links, `description`, `county` (100% missing), `size` (72% missing), `lat`/`long`.
- `VIN` is kept for now, because step 4 needs it.
- Text is stripped and lowercased so "Ford" and "ford" match. `VIN` is uppercased.
- `posting_date` becomes a real datetime.

In [ ]:
df = df.drop(columns=["id", "url", "region_url", "image_url", "description",
                      "county", "size", "lat", "long"])

text_cols = ["region", "manufacturer", "model", "condition", "cylinders", "fuel",
             "title_status", "transmission", "drive", "type", "paint_color", "state"]
for c in text_cols:
    df[c] = df[c].str.strip().str.lower()

df["VIN"] = df["VIN"].str.strip().str.upper()
df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)

## 3. Invalid values

Rows outside the limits are removed. Missing years and odometers are kept for now and handled in step 5.

Domain limits are used instead of IQR, because price and odometer are very right-skewed and IQR would remove thousands of normal listings.

In [ ]:
df = df[df["price"].between(PRICE_MIN, PRICE_MAX)]
print("After price filter:", len(df))

df = df[df["year"].between(YEAR_MIN, YEAR_MAX) | df["year"].isna()]
print("After year filter:", len(df))

df = df[(df["odometer"] <= ODO_MAX) | df["odometer"].isna()]
print("After odometer filter:", len(df))

## 4. Duplicates

A duplicate is **the same vehicle posted again**, not a repeated `id`.

- **Has a VIN (17 characters):** the same VIN means the same vehicle.
- **No VIN:** the same manufacturer, model, year, odometer, price, state and other attributes means the same vehicle.
- The **newest posting is kept**, because its price is the most current.

In [ ]:
df = df.sort_values("posting_date")          # oldest first, so keep="last" keeps the newest

key = ["manufacturer", "model", "year", "odometer", "price", "state",
       "condition", "cylinders", "fuel", "transmission", "drive", "type", "paint_color"]

has_vin = df["VIN"].str.len() == 17
with_vin = df[has_vin].drop_duplicates(subset="VIN", keep="last")
no_vin = df[~has_vin].drop_duplicates(subset=key, keep="last")

print("Removed by VIN:", has_vin.sum() - len(with_vin))
print("Removed by attributes:", (~has_vin).sum() - len(no_vin))

df = pd.concat([with_vin, no_vin]).drop(columns="VIN")

## 5. Missing values

- Rows with fewer than 6 of 12 core fields filled are dropped (filling them would invent a car).
- Rows missing `year` or `odometer` are dropped (only about 0.3% and 1%).
- Missing categories become `"unknown"`.

In [ ]:
core = ["year", "odometer", "manufacturer", "model", "condition", "cylinders",
        "fuel", "title_status", "transmission", "drive", "type", "paint_color"]

df = df.dropna(subset=core, thresh=6)
df = df.dropna(subset=["year", "odometer"])
df["year"] = df["year"].astype(int)
df[text_cols] = df[text_cols].fillna("unknown")

## 6. Final check

In [ ]:
df = df.reset_index(drop=True)
print("Final shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
df[["price", "year", "odometer"]].describe().round(0)